# 01 数据探索
加载 FU（燃油期货）1分钟K线，探索价格结构、成交量与持仓量。

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.data.universe import ContinuousContract

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
loader = FuturesDataLoader(DATA_DIR)

## 1. 可用合约一览

In [ ]:
contracts = loader.list_contracts('FU')
summary = pd.DataFrame([
    {'合约': c.contract_id, 'tqsdk代码': c.tqsdk_symbol,
     '交割年': c.year, '交割月': c.month}
    for c in contracts
])
print(f'共 {len(summary)} 个合约，{summary["交割年"].min()} 年 ~ {summary["交割年"].max()} 年')
summary.tail(10)

## 2. 单合约 K 线

In [ ]:
df = loader.load('FU2210', start='2022-01-01', end='2022-10-31')
print(f'行数: {len(df):,}   时间范围: {df.index[0]} ~ {df.index[-1]}')
df.describe().round(2)

In [ ]:
# 日线聚合（便于可视化）
daily = df.resample('1D').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
    'volume': 'sum', 'amount': 'sum', 'open_interest': 'last'
}).dropna()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=['K 线 (日)', '成交量', '持仓量'],
                    row_heights=[0.5, 0.25, 0.25], vertical_spacing=0.05)

fig.add_trace(go.Candlestick(
    x=daily.index, open=daily['open'], high=daily['high'],
    low=daily['low'], close=daily['close'], name='FU2210'
), row=1, col=1)

fig.add_trace(go.Bar(x=daily.index, y=daily['volume'],
                     marker_color='steelblue', name='成交量'), row=2, col=1)

fig.add_trace(go.Scatter(x=daily.index, y=daily['open_interest'],
                         line=dict(color='orange'), name='持仓量'), row=3, col=1)

fig.update_layout(title='FU2210 日线概览', height=700, xaxis_rangeslider_visible=False)
fig.show()

## 3. 日内分钟分布

In [ ]:
# 每分钟平均成交量（查看交易时段结构）
df['time_of_day'] = df.index.time
intraday_vol = df.groupby('time_of_day')['volume'].mean().reset_index()
intraday_vol['time_str'] = intraday_vol['time_of_day'].astype(str)

fig = px.bar(intraday_vol, x='time_str', y='volume',
             title='日内各分钟平均成交量 (FU2210 2022全年)',
             labels={'time_str': '时间', 'volume': '平均成交量(手)'})
fig.update_xaxes(tickangle=45, nticks=20)
fig.show()

## 4. 连续合约（主力，后复权）

In [ ]:
cc = ContinuousContract(loader, 'FU', adjust='back')
cont = cc.build(start='2020-01-01', end='2022-12-31')

# 日线聚合
cont_daily = cont.resample('1D').agg({
    'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
    'volume': 'sum', 'open_interest': 'last', 'contract': 'last'
}).dropna()

# 标记换月点
roll_dates = cont_daily[cont_daily['contract'] != cont_daily['contract'].shift(1)].index

fig = go.Figure()
fig.add_trace(go.Scatter(x=cont_daily.index, y=cont_daily['close'],
                         name='主力连续(后复权)', line=dict(color='#1f77b4')))
for rd in roll_dates:
    fig.add_vline(x=rd, line_dash='dash', line_color='gray', opacity=0.4)

fig.update_layout(title='FU 主力连续合约 2020-2022（灰线=换月）', height=450)
fig.show()

print(f'换月次数: {len(roll_dates)}')
print('涉及合约:', cont_daily['contract'].unique())

## 5. 收益率分布

In [ ]:
import numpy as np

ret = df['close'].pct_change().dropna()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['1分钟收益率分布', 'QQ图（vs 正态）'])

fig.add_trace(go.Histogram(x=ret, nbinsx=200, name='收益率',
                            marker_color='steelblue', opacity=0.7), row=1, col=1)

# QQ 图
from scipy import stats
qq = stats.probplot(ret)
fig.add_trace(go.Scatter(x=qq[0][0], y=qq[0][1], mode='markers',
                          name='样本分位数', marker=dict(size=3)), row=1, col=2)
fig.add_trace(go.Scatter(x=qq[0][0],
                          y=qq[1][1] + qq[1][0] * qq[0][0],
                          mode='lines', name='正态参考线',
                          line=dict(color='red')), row=1, col=2)

fig.update_layout(height=400, title='FU2210 收益率统计特征')
fig.show()

print(f'偏度: {ret.skew():.4f}   峰度: {ret.kurtosis():.4f}')
print(f'年化波动率: {ret.std() * np.sqrt(240*250):.2%}')